# 03 | Combined model

**Question:** Does combining NCAA and combine data improve prediction?

**Shared outcome:** NBA points per game over the three post-combine seasons. **Common sample:** players with matched NCAA data. **Same split:** 80/20, seed 42, stratified by position. No API calls here.

**Steps:** Run cells top to bottom, inspect all five assumptions, and report the equation with estimated coefficients plus MSE and R². If a violation needs fixing, edit `FIX_FORMULA` near the end and document why.

**Matching policy:** punctuation/spacing fixes and explicit aliases; ambiguous same-name players and multi-school source conflicts are excluded. NCAA predictors use the season ending in the recorded combine year, even if the player later returns to college. Rerun `00_build_dataset.ipynb` after changing these rules.

In [ ]:
MODEL='combined'
PREDICTORS=['ncaa_ppg', 'ncaa_fg_pct', 'wingspan_in', 'vertical_max_in']
FORMULA='ppg_3yr ~ ncaa_ppg + ncaa_fg_pct + wingspan_in + vertical_max_in + C(position_group, Treatment(reference="Guard"))'

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

ROOT=Path.cwd()
DATA=ROOT/'data'
REPORT=ROOT/'reports'
REPORT.mkdir(exist_ok=True)
X=pd.read_csv(DATA/'predraft_features.csv')
y=pd.read_csv(DATA/'nba_outcomes.csv')
assert X.player_id.is_unique and y.player_id.is_unique
assert set(X.player_id) == set(y.player_id)
required = {'ncaa_season','ncaa_team_count','ncaa_candidate_count','ncaa_match_method'}
if not required.issubset(X.columns):
    raise ValueError('Rebuild data with the updated 00_build_dataset.ipynb first.')
sample=(X.merge(y[['player_id','ppg_3yr']],on='player_id',validate='one_to_one')
          .query('ncaa_match_status == "matched"').sort_values('player_id').reset_index(drop=True))
sample=sample[sample.position_group.isin(['Guard','Forward','Center'])].reset_index(drop=True)
assert sample.ncaa_candidate_count.eq(1).all(), 'Ambiguous names must be excluded.'
assert sample.ncaa_team_count.eq(1).all(), 'Conflicting NCAA teams must be excluded.'
assert sample.ncaa_season.eq(sample.combine_year).all(), 'Use college stats available at the combine.'
if len(sample)<90: raise ValueError(f'Only {len(sample)} matched NBA/NCAA players; inspect the match audit before modeling.')
if sample.position_group.value_counts().min()<3: raise ValueError('Insufficient players in at least one position group.')
tr_ids,te_ids=train_test_split(sample.index,test_size=.20,random_state=42,stratify=sample.position_group)
train,test=sample.loc[tr_ids].copy(),sample.loc[te_ids].copy()
print('Matched sample:',len(sample),'| Train:',len(train),'| Test:',len(test))

print('Missing quantitative measurements in matched training sample (%)')
print((100*train[PREDICTORS].isna().mean()).round(1).to_string())

# Impute using the training data ONLY. Keep identical player IDs across all models.
for c in PREDICTORS:
    train[c]=pd.to_numeric(train[c],errors='coerce')
    test[c]=pd.to_numeric(test[c],errors='coerce')
    if train[c].notna().sum()<20:
        raise ValueError(f'Too few measured values for {c}; revisit your feature specification.')
    med=train[c].median()
    train[c]=train[c].fillna(med)
    test[c]=test[c].fillna(med)
if 'vertical_max_in' in PREDICTORS:
    vmean=train.vertical_max_in.mean()
    train['vertical_centered']=train.vertical_max_in-vmean
    test['vertical_centered']=test.vertical_max_in-vmean
print('Predictors:',PREDICTORS)

# Explore training predictors without inspecting the holdout.
print('Quantitative predictor correlations (imputed training sample):')
display(train[PREDICTORS].corr().round(2))

In [ ]:
# Fit on training data only. If needed, edit FIX_FORMULA based on these diagnostics, then rerun.
initial=smf.ols(FORMULA, data=train).fit()
print(initial.summary().tables[1])

def diagnostics(fit,label):
    residual=fit.resid
    fitted=fit.fittedvalues
    fig,axs=plt.subplots(1,3,figsize=(14,3.6))
    axs[0].scatter(fitted,residual,alpha=.55,s=17)
    axs[0].axhline(0,color='black',linestyle='--',linewidth=1)
    axs[0].set(xlabel='Fitted NBA PPG',ylabel='Residual (PPG)',title='Linearity and variance')
    sm.qqplot(residual,line='45',fit=True,ax=axs[1]);axs[1].set(title='Normal Q-Q')
    s=np.std(residual,ddof=1)
    axs[2].scatter(fitted,np.sqrt(np.abs(residual/s)),alpha=.55,s=17)
    axs[2].set(xlabel='Fitted NBA PPG',ylabel='√|standardized residual|',title='Scale-location')
    fig.tight_layout()
    fig.savefig(REPORT/f'{MODEL}_{label}_diagnostics.png',dpi=160)
    plt.show()
    design=pd.DataFrame(fit.model.exog,columns=fit.model.exog_names)
    vif=pd.DataFrame([{'variable':col,'VIF':variance_inflation_factor(design.to_numpy(),i)}
            for i,col in enumerate(design.columns) if col!='Intercept'])
    print('VIF (course guideline < 5):')
    display(vif)
    return vif

initial_vif=diagnostics(initial,'initial')
print('Independence: one row per unique player; cohort-level correlations remain possible.')
print('If a serious issue appears, set FIX_FORMULA below and explain the change in slides.')

In [ ]:
# OPTIONAL FIX. Keep None unless justified by an actual training residual or VIF diagnostic.
# Example for Member 1: remove height if highly collinear with wingspan.
FIX_FORMULA=None
final=initial if FIX_FORMULA is None else smf.ols(FIX_FORMULA,data=train).fit()
if FIX_FORMULA is not None:
    final_vif=diagnostics(final,'fixed')
else:
    final_vif=initial_vif

# Holdout evaluation once, using the common split (seed 42, stratified by position).
pred=final.predict(test)
mse=mean_squared_error(test.ppg_3yr,pred)
r2=r2_score(test.ppg_3yr,pred)
print(f'\nTEST MSE: {mse:.3f} PPG² | TEST R²: {r2:.3f}')
print('Final equation:',final.model.formula)
coef=pd.DataFrame({'term':final.params.index,'estimate':final.params.values,
                   'p_value':final.pvalues.values})
display(coef)
coef.to_csv(REPORT/f'{MODEL}_coefficients.csv',index=False)
summary=pd.DataFrame([{'model':MODEL,'train_n':len(train),'test_n':len(test),
                       'test_mse':mse,'test_r2':r2,
                       'formula':final.model.formula,
                       'max_vif':final_vif.VIF.max()}])
summary.to_csv(REPORT/f'{MODEL}_metrics.csv',index=False)
print('Saved metrics, coefficients, and diagnostic figure(s) to reports/.')